Just playing around with the APi

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd, sklearn, requests, matplotlib
# checking python path
print(sys.executable)
# checking versions 
print(pd.__version__, sklearn.__version__)

In [ ]:
# fetching data, retrieves all matches from 2020 season of the  matchday
r = requests.get(f'https://api.openligadb.de/getmatchdata/bl1/2020/', timeout=10)
r.raise_for_status()
matches = r.json()
len(matches), matches

In [ ]:
m = matches[0]
m['team2']['teamName'], m['group']['groupOrderID'], m['matchIsFinished']



In [ ]:
for result in m['matchResults']:
    print(result['pointsTeam1'], result['pointsTeam2'])


def full_time(match):
    for result in match['matchResults']:
        # 2 is the score after full time, 1 is halftime
        if result['resultTypeID'] == 2:
            return result['pointsTeam1'], result['pointsTeam2']
    return None, None

full_time(m)
            

In [ ]:
home_goals, away_goals = full_time(m) # unpacks the tuple
row = {
    'season': m['leagueSeason'],
    'matchday': m['group']['groupOrderID'],
    'date': m['matchDateTime'],
    'home_team': m['team1']['teamName'],
    'away_team': m['team2']['teamName'],
    'home_goals': home_goals,
    'away_goals': away_goals,
    'finished': m['matchIsFinished']
}
row




In [ ]:
rows = []
for m in matches:
    home_goals, away_goals = full_time(m)
    rows.append({
        'season': m['leagueSeason'],
        'matchday': m['group']['groupOrderID'],
        'date': m['matchDateTime'],
        'home_team': m['team1']['teamName'],
        'away_team': m['team2']['teamName'],
        'home_goals': home_goals,
        'away_goals': away_goals,
        'finished': m['matchIsFinished']
    })

df = pd.DataFrame(rows)
df

In [ ]:
df.shape 
# df.matchday.nunique()
# df.finished.all()

In [ ]:
bvb_home = df[df.home_team == 'Borussia Dortmund'] # rows where bvb is the home team
(bvb_home.home_goals > bvb_home.away_goals).sum() # bvb number of wins home

In [ ]:
bvb_away = df[df.away_team == 'Borussia Dortmund'] # rows where bvb is the away team
(bvb_away.home_goals < bvb_away.away_goals).sum() # bvb number of wins away

In [ ]:
bvb_draw_home = df[df.home_team == 'Borussia Dortmund']
(bvb_draw_home.home_goals == bvb_draw_home.away_goals).sum() # bvb number of draws
bvb_draw_away = df[df.away_team == 'Borussia Dortmund']
(bvb_draw_away.home_goals == bvb_draw_away.away_goals).sum() # bvb number of draws away



In [ ]:
bvb_home = df[df.home_team == 'Borussia Dortmund']
bvb_away = df[df.away_team == 'Borussia Dortmund']

home_wins = (bvb_home.home_goals > bvb_home.away_goals).sum() # bvb number of wins home
away_wins = (bvb_away.home_goals < bvb_away.away_goals).sum() # bvb number of wins away
home_draws = (bvb_draw_home.home_goals == bvb_draw_home.away_goals).sum() # bvb number of draws home
away_draws = (bvb_draw_away.home_goals == bvb_draw_away.away_goals).sum() # bvb number of draws away

draws = home_draws + away_draws

bvb_points_2020 = ((home_wins + away_wins) * 3) + draws 
bvb_points_2020




this is teh expected score for the home team, 0 and 1

In [ ]:
HOME_ADV = 60 # home advantage in ELO
def expected_score(r_home, r_away):
    return  1 / (1+10 ** ((r_away - r_home - HOME_ADV) / 400))

expected_score(1500, 1500) # 0.585 equal but home advantage
expected_score(1500, 1560) # 0.57
expected_score(1700, 1400) # 0.8 high strong home team

In [ ]:
HOME_ADVANTAGE = 60   # Elo points added to the home team
K_FACTOR = 20         # how far one match can move a rating

def expected_score(home_rating, away_rating):
    return 1 / (1 + 10 ** ((away_rating - home_rating - HOME_ADVANTAGE) / 400))

def actual_score(home_goals, away_goals):
    # 1 for a home win, 0.5 for a draw, 0 for an away win
    if home_goals > away_goals:
        return 1
    elif home_goals == away_goals:
        return 0.5
    else:
        return 0

def update(home_rating, away_rating, home_goals, away_goals):
    expected = expected_score(home_rating, away_rating)
    actual = actual_score(home_goals, away_goals)
    rating_change = K_FACTOR * (actual - expected)
    return home_rating + rating_change, away_rating - rating_change


In [ ]:
print(update(1500, 1500, 2, 0))       # about (1508.3, 1491.7)
print(update(1500, 1500, 0, 0))       # about (1498.3, 1501.7)
print(sum(update(1600, 1450, 1, 3)))  # 3050.0


In [ ]:
from src.features import add_elo, LEAGUE_AVERAGE
from src.data import load_all

matches = add_elo(load_all())
matches[['date', 'home_team', 'away_team', 'home_elo_before', 'away_elo_before']].tail()



In [ ]:
print(len(matches), matches['home_elo_before'].isna().sum())

In [ ]:
matches.groupby('home_team')['home_elo_before'].last().sort_values(ascending=False).head(5)

In [ ]:
bvb = matches[
    (matches.home_team == 'Borussia Dortmund') |
    (matches.away_team == 'Borussia Dortmund')
].copy()

bvb['bvb_elo'] = bvb.home_elo_before.where(
    bvb.home_team == 'Borussia Dortmund',
    bvb.away_elo_before,
)
bvb = bvb[bvb.season >= 2003]
# they sold erling haaland in 2022
bvb_2022 = bvb[bvb.season == 2022]
# they sold jude bellingham in 2023
bvb_2023 = bvb[bvb.season == 2023]
bvb_2012 = bvb[bvb.season == 2012]
# bvb_2012.head(1)

In [ ]:
import matplotlib.pyplot as plt
played = bvb[bvb.finished]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(played.date, played.bvb_elo)
ax.axhline(LEAGUE_AVERAGE, color='black', linestyle='--')
ax.set_title('BVB ELO rating in 2003 to today')
ax.set_xlabel('Date')
ax.set_ylabel('ELO rating')
plt.show()

In [ ]:
print(len(matches), matches['home_elo_before'].isna().sum())

In [ ]:
# get the average of the last 5 games as a second feature to the ELO.
# This is also a good way to handle the missing values using the form of each team.
home_rows = pd.DataFrame({
    'date': matches.date,
    'team': matches.home_team,
    'points': ((matches.home_goals > matches.away_goals) * 3 + (matches.home_goals == matches.away_goals) * 1).where(matches.finished), # 3 for a win, 1 for a draw, 0 for a loss
    'finished': matches.finished,
    'season': matches.season
})
away_rows = pd.DataFrame({
    'date': matches.date,
    'team': matches.away_team,
    'points': ((matches.away_goals > matches.home_goals) * 3 + (matches.away_goals == matches.home_goals) * 1).where(matches.finished),
    'finished': matches.finished,
    'season': matches.season
})

team_matches = pd.concat([home_rows, away_rows], ignore_index=True).sort_values('date')

finished_rows = team_matches[team_matches.finished]

team_matches['form_after'] = (
    finished_rows
    .groupby('team')          # one team at a time
    .points
    .transform(lambda team_points: team_points.rolling(5).mean())
)

In [ ]:
team_matches.points.isna().sum()

In [ ]:
bvb_matches = team_matches[(team_matches.team == 'Borussia Dortmund') & (team_matches.finished) & (team_matches.season == 2020)]

bvb_matches.head(10)